# Transformer Models — Zero-Shot and Fine-Tuned
Sentiment classification using XLM-RoBERTa, compared against classical ML baselines.

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.metrics import classification_report, accuracy_score, cohen_kappa_score, confusion_matrix
import os
import warnings
warnings.filterwarnings('ignore')

device = 0 if torch.cuda.is_available() else -1
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {"GPU" if device == 0 else "CPU"}')

CUDA available: False
Device: CPU


## 1. Load Gold Test Set

In [2]:
test_df = pd.read_csv('../data/NLP_Dataset_preparation/gold_labeled.csv')
y_test = test_df['human_label']
print(f'Test rows: {len(test_df)}')
print(test_df['human_label'].value_counts())

Test rows: 402
human_label
neutral     241
negative    124
positive     37
Name: count, dtype: int64


## 2. Zero-Shot — cardiffnlp/twitter-xlm-roberta-base-sentiment
No training. Pretrained for sentiment on multilingual social media text. This tests whether a generic multilingual sentiment model already works on our domain without seeing our data.

In [3]:
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

zero_shot_pipe = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=device,
    truncation=True,
    max_length=512
)

label_map = {
    'positive': 'positive',
    'neutral': 'neutral',
    'negative': 'negative',
    'LABEL_0': 'negative',
    'LABEL_1': 'neutral',
    'LABEL_2': 'positive',
}

texts = test_df['body'].astype(str).tolist()
raw_preds = zero_shot_pipe(texts, batch_size=16)
zero_shot_preds = [label_map.get(p['label'].lower(), label_map.get(p['label'], 'neutral')) for p in raw_preds]

print('Sample predictions:')
for i in range(5):
    print(f"  [{zero_shot_preds[i]:<8}] (actual: {y_test.iloc[i]:<8}) {texts[i][:80]}")

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Sample predictions:
  [negative] (actual: negative) I did not agree that we are above average as malaysians are still struggling to 
  [neutral ] (actual: neutral ) What is the inflation rate used? 1625 per month to 4434
  [positive] (actual: negative) Got some dulu kata harga minyak takda masalah, if naik untung kerjaan naik so bo
  [neutral ] (actual: neutral ) PPP isn't exactly a measure of purchasing power over time... PPP is a measure be
  [negative] (actual: negative) > 1 year of work in a developed European nation is worth 4 years here >  > . >  


## 3. Zero-Shot Evaluation

In [4]:
acc_zs = accuracy_score(y_test, zero_shot_preds)
kappa_zs = cohen_kappa_score(y_test, zero_shot_preds)
report_zs = classification_report(y_test, zero_shot_preds, target_names=['negative','neutral','positive'], output_dict=True)

print(f'XLM-R Zero-Shot (cardiffnlp)')
print(f'Accuracy: {acc_zs*100:.1f}%   Kappa: {kappa_zs:.3f}')
print(classification_report(y_test, zero_shot_preds, target_names=['negative','neutral','positive']))

print('\nConfusion matrix (rows=actual, cols=predicted):')
labels = ['negative','neutral','positive']
cm = confusion_matrix(y_test, zero_shot_preds, labels=labels)
print(f"{'':>10}" + "".join(f"{l:>10}" for l in labels))
for i, l in enumerate(labels):
    print(f"{l:>10}" + "".join(f"{cm[i][j]:>10}" for j in range(len(labels))))

XLM-R Zero-Shot (cardiffnlp)
Accuracy: 48.3%   Kappa: 0.169
              precision    recall  f1-score   support

    negative       0.39      0.82      0.53       124
     neutral       0.77      0.37      0.49       241
    positive       0.15      0.11      0.13        37

    accuracy                           0.48       402
   macro avg       0.44      0.43      0.38       402
weighted avg       0.59      0.48      0.47       402


Confusion matrix (rows=actual, cols=predicted):
            negative   neutral  positive
  negative       102        13         9
   neutral       140        88        13
  positive        19        14         4


## 4. Save Zero-Shot Results

In [5]:
os.makedirs('../results', exist_ok=True)
RESULTS_FILE = '../results/model_results.csv'

new_row = pd.DataFrame([{
    'model': 'XLM-R Zero-Shot (cardiffnlp)',
    'status': 'transformer_zero_shot',
    'accuracy': acc_zs,
    'kappa': kappa_zs,
    'neg_f1': report_zs['negative']['f1-score'],
    'neu_f1': report_zs['neutral']['f1-score'],
    'pos_f1': report_zs['positive']['f1-score'],
}])

existing = pd.read_csv(RESULTS_FILE)
existing = existing[existing['model'] != 'XLM-R Zero-Shot (cardiffnlp)']
combined = pd.concat([existing, new_row], ignore_index=True)
combined.to_csv(RESULTS_FILE, index=False)
print(combined.to_string(index=False))

                       model                status  accuracy    kappa   neg_f1   neu_f1   pos_f1
        Naive Bayes + TF-IDF            final_best  0.641791 0.372733 0.556452 0.738462 0.415842
           Naive Bayes + BoW baseline_not_selected  0.611940 0.322025 0.542751 0.701124 0.377778
                SVM + TF-IDF baseline_not_selected  0.676617 0.357124 0.550218 0.765957 0.379310
                   SVM + BoW            final_best  0.684080 0.366489 0.556054 0.772467 0.379310
Logistic Regression + TF-IDF            final_best  0.669154 0.364884 0.571429 0.754032 0.380952
         Naive Bayes (tuned)  tuned_underperformed  0.614428 0.337353 0.520661 0.717149 0.407080
                 SVM (tuned)  tuned_underperformed  0.634328 0.335761 0.549801 0.723404 0.385542
 Logistic Regression (tuned)  tuned_underperformed  0.644279 0.342664 0.555985 0.729958 0.394366
XLM-R Zero-Shot (cardiffnlp) transformer_zero_shot  0.482587 0.168521 0.529870 0.494382 0.126984
